# Iterative Supervised Thematic Analysis (Notebook)

This notebook supports an iterative qualitative workflow:
1. Create a random coding sample (10 or 20 rows) for one question.
2. Human-code multi-label themes in the sample sheet.
3. Train a supervised model from those labels.
4. Run unsupervised clustering to discover additional themes.

## Important

- Labels are **not mutually exclusive**.
- Use `|` between multiple labels in `human_labels`.
- You can repeat this cycle as many times as needed.

In [ ]:
%pip install -r ../requirements.txt

In [ ]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "iterative_supervised_thematic.py").exists():
    ROOT = cwd
elif (cwd.parent / "iterative_supervised_thematic.py").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not find iterative_supervised_thematic.py")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from analyze_survey import load_yaml
from iterative_supervised_thematic import create_coding_sample, train_and_apply

In [ ]:
CONFIG_PATH = ROOT / "config" / "iterative_supervised.example.yaml"
config = load_yaml(CONFIG_PATH)
config

In [ ]:
# Optional overrides before creating sample
# config["iterative_supervised"]["question_value"] = "What did you like least?"
# config["iterative_supervised"]["sample_size"] = 20
# config["iterative_supervised"]["output_dir"] = str(ROOT / "output_iterative")

In [ ]:
# STEP 1: Create coding sample sheet.
create_coding_sample(config)
print("Sample created.")

## Manual step

Open `output_iterative/coding_sample.csv` and fill `human_labels`.

Examples:
- `Usability issue`
- `Price concern|Usability issue`
- `Feature request`

Then return and run the next cell.

In [ ]:
# STEP 2: Train supervised model and run unsupervised discovery.
train_and_apply(config)
print("Train + unsupervised pass complete.")

In [ ]:
out_dir = Path(config["iterative_supervised"]["output_dir"]).expanduser().resolve()
pred = pd.read_csv(out_dir / "iterative_predictions.csv")
cluster_summary = pd.read_csv(out_dir / "iterative_cluster_summary.csv")
label_summary = pd.read_csv(out_dir / "iterative_label_summary.csv")
label_cluster = pd.read_csv(out_dir / "iterative_label_cluster_matrix.csv")

print("Output directory:", out_dir)
display(pred.head(20))
display(cluster_summary)
display(label_summary)
display(label_cluster.head(20))

## Iteration tips

1. Review `UNCERTAIN_REVIEW` rows first.
2. Improve label definitions and add more coded rows.
3. Re-run train step.
4. Compare label distributions and cluster patterns over time.